In [1]:
"""
Data ingestion and quality assessment notebook.

Purpose:
- Load the bank churn and marketing A/B testing datasets
- Inspect their structure
- Check for data quality issues
- Standardize column naming
- Prepare them for downstream EDA and modeling
"""

from pathlib import Path
import pandas as pd
import numpy as np

In [2]:
"""
Configure pandas display settings for easier inspection during analysis.
"""

pd.set_option("display.max_columns", None)
pd.set_option("display.width", 120)
pd.set_option("display.max_colwidth", 100)

In [3]:
"""
Define project paths.

This approach makes the notebook more robust and easier to reuse
across different machines, as paths are built dynamically.
"""

PROJECT_ROOT = Path("..").resolve()
DATA_RAW_DIR = PROJECT_ROOT / "data" / "raw"

CHURN_DATA_PATH = DATA_RAW_DIR / "bank_churn.csv"
AB_DATA_PATH = DATA_RAW_DIR / "marketing_ab_test.csv"

print("Project root:", PROJECT_ROOT)
print("Churn data path exists:", CHURN_DATA_PATH.exists())
print("A/B data path exists:", AB_DATA_PATH.exists())

Project root: C:\retention_ds\retention_ds
Churn data path exists: True
A/B data path exists: True


In [4]:
"""
Helper functions for loading data and performing reusable quality checks.
"""

def load_csv_data(file_path: Path) -> pd.DataFrame:
    """
    Load a CSV file into a pandas DataFrame.

    Parameters
    ----------
    file_path : Path
        The path to the CSV file.

    Returns
    -------
    pd.DataFrame
        The loaded dataset.

    Raises
    ------
    FileNotFoundError
        If the file does not exist.
    """
    if not file_path.exists():
        raise FileNotFoundError(f"File not found: {file_path}")
    
    return pd.read_csv(file_path)


def standardize_column_names(df: pd.DataFrame) -> pd.DataFrame:
    """
    Standardize column names for consistency.

    Actions performed:
    - Strip leading/trailing spaces
    - Convert to lowercase
    - Replace spaces with underscores

    Parameters
    ----------
    df : pd.DataFrame
        Input DataFrame.

    Returns
    -------
    pd.DataFrame
        DataFrame with standardized column names.
    """
    df = df.copy()
    df.columns = (
        df.columns
        .str.strip()
        .str.lower()
        .str.replace(" ", "_", regex=False)
    )
    return df


def dataset_overview(df: pd.DataFrame, dataset_name: str) -> None:
    """
    Print a high-level overview of a dataset.

    Parameters
    ----------
    df : pd.DataFrame
        The dataset to inspect.
    dataset_name : str
        A friendly name for the dataset.
    """
    print(f"\n{'=' * 80}")
    print(f"{dataset_name.upper()} OVERVIEW")
    print(f"{'=' * 80}")
    print(f"Shape: {df.shape}")
    print("\nColumns:")
    print(list(df.columns))
    print("\nData types:")
    print(df.dtypes)
    print("\nFirst 5 rows:")
    display(df.head())


def missing_values_summary(df: pd.DataFrame) -> pd.DataFrame:
    """
    Create a summary of missing values for each column.

    Parameters
    ----------
    df : pd.DataFrame
        The dataset to inspect.

    Returns
    -------
    pd.DataFrame
        A summary table with missing counts and percentages.
    """
    summary = pd.DataFrame({
        "missing_count": df.isnull().sum(),
        "missing_pct": (df.isnull().mean() * 100).round(2)
    }).sort_values(by="missing_count", ascending=False)
    
    return summary


def duplicate_summary(df: pd.DataFrame) -> dict:
    """
    Count duplicate rows in a dataset.

    Parameters
    ----------
    df : pd.DataFrame
        The dataset to inspect.

    Returns
    -------
    dict
        Dictionary containing duplicate row count and percentage.
    """
    duplicate_count = df.duplicated().sum()
    duplicate_pct = round((duplicate_count / len(df)) * 100, 2)
    
    return {
        "duplicate_count": duplicate_count,
        "duplicate_pct": duplicate_pct
    }


def categorical_value_summary(df: pd.DataFrame, columns: list[str]) -> None:
    """
    Display value counts for selected categorical columns.

    Parameters
    ----------
    df : pd.DataFrame
        The dataset containing the columns.
    columns : list[str]
        List of categorical column names to summarize.
    """
    for column in columns:
        print(f"\nValue counts for '{column}':")
        print(df[column].value_counts(dropna=False))

In [5]:
"""
Load the raw datasets into memory.
"""

churn_df = load_csv_data(CHURN_DATA_PATH)
ab_df = load_csv_data(AB_DATA_PATH)

print("Datasets loaded successfully.")

Datasets loaded successfully.


In [6]:
"""
Standardize column names early so that all downstream code uses
consistent and predictable naming.
"""

churn_df = standardize_column_names(churn_df)
ab_df = standardize_column_names(ab_df)

print("Standardized churn columns:")
print(churn_df.columns.tolist())

print("\nStandardized A/B testing columns:")
print(ab_df.columns.tolist())

Standardized churn columns:
['rownumber', 'customerid', 'surname', 'creditscore', 'geography', 'gender', 'age', 'tenure', 'balance', 'numofproducts', 'hascrcard', 'isactivemember', 'estimatedsalary', 'exited']

Standardized A/B testing columns:
['unnamed:_0', 'user_id', 'test_group', 'converted', 'total_ads', 'most_ads_day', 'most_ads_hour']


In [7]:
"""
Inspect the structure of both datasets.
"""

dataset_overview(churn_df, "Bank Churn Dataset")
dataset_overview(ab_df, "Marketing A/B Testing Dataset")


BANK CHURN DATASET OVERVIEW
Shape: (10000, 14)

Columns:
['rownumber', 'customerid', 'surname', 'creditscore', 'geography', 'gender', 'age', 'tenure', 'balance', 'numofproducts', 'hascrcard', 'isactivemember', 'estimatedsalary', 'exited']

Data types:
rownumber            int64
customerid           int64
surname             object
creditscore          int64
geography           object
gender              object
age                  int64
tenure               int64
balance            float64
numofproducts        int64
hascrcard            int64
isactivemember       int64
estimatedsalary    float64
exited               int64
dtype: object

First 5 rows:


,rownumber,customerid,surname,creditscore,geography,gender,age,tenure,balance,numofproducts,hascrcard,isactivemember,estimatedsalary,exited
0,1,15634602,Hargrave,619,France,Female,42,2,0.00,1,1,1,101348.88,1
1,2,15647311,Hill,608,Spain,Female,41,1,83807.86,1,0,1,112542.58,0
2,3,15619304,Onio,502,France,Female,42,8,159660.80,3,1,0,113931.57,1
3,4,15701354,Boni,699,France,Female,39,1,0.00,2,0,0,93826.63,0
4,5,15737888,Mitchell,850,Spain,Female,43,2,125510.82,1,1,1,79084.10,0



MARKETING A/B TESTING DATASET OVERVIEW
Shape: (588101, 7)

Columns:
['unnamed:_0', 'user_id', 'test_group', 'converted', 'total_ads', 'most_ads_day', 'most_ads_hour']

Data types:
unnamed:_0        int64
user_id           int64
test_group       object
converted          bool
total_ads         int64
most_ads_day     object
most_ads_hour     int64
dtype: object

First 5 rows:


,unnamed:_0,user_id,test_group,converted,total_ads,most_ads_day,most_ads_hour
0,0,1069124,ad,False,130,Monday,20
1,1,1119715,ad,False,93,Tuesday,22
2,2,1144181,ad,False,21,Tuesday,18
3,3,1435133,ad,False,355,Tuesday,10
4,4,1015700,ad,False,276,Friday,14


In [8]:
"""
Assess missing values in each dataset.

This helps identify whether imputation, dropping, or special handling
will be needed later.
"""

print("Churn dataset missing values summary:")
display(missing_values_summary(churn_df))

print("\nA/B testing dataset missing values summary:")
display(missing_values_summary(ab_df))

Churn dataset missing values summary:


,missing_count,missing_pct
rownumber,0,0.0
customerid,0,0.0
surname,0,0.0
creditscore,0,0.0
geography,0,0.0
gender,0,0.0
age,0,0.0
tenure,0,0.0
balance,0,0.0
numofproducts,0,0.0



A/B testing dataset missing values summary:


,missing_count,missing_pct
unnamed:_0,0,0.0
user_id,0,0.0
test_group,0,0.0
converted,0,0.0
total_ads,0,0.0
most_ads_day,0,0.0
most_ads_hour,0,0.0


In [9]:
"""
Check for duplicate rows in both datasets.

Duplicate records can distort analysis and model performance if not handled properly.
"""

churn_duplicates = duplicate_summary(churn_df)
ab_duplicates = duplicate_summary(ab_df)

print("Churn duplicate summary:", churn_duplicates)
print("A/B testing duplicate summary:", ab_duplicates)

Churn duplicate summary: {'duplicate_count': np.int64(0), 'duplicate_pct': np.float64(0.0)}
A/B testing duplicate summary: {'duplicate_count': np.int64(0), 'duplicate_pct': np.float64(0.0)}


In [10]:
"""
Inspect the key target/outcome variables.

For the churn dataset:
- exited is the churn target

For the A/B testing dataset:
- converted is the experimental outcome
- test_group is the treatment assignment
"""

print("Churn target distribution (exited):")
print(churn_df["exited"].value_counts(dropna=False))
print("\nChurn target distribution (percentage):")
print((churn_df["exited"].value_counts(normalize=True) * 100).round(2))

print("\n" + "=" * 80)

print("A/B conversion distribution (converted):")
print(ab_df["converted"].value_counts(dropna=False))
print("\nA/B conversion distribution (percentage):")
print((ab_df["converted"].value_counts(normalize=True) * 100).round(2))

print("\n" + "=" * 80)

print("A/B treatment group distribution (test_group):")
print(ab_df["test_group"].value_counts(dropna=False))
print("\nA/B treatment group distribution (percentage):")
print((ab_df["test_group"].value_counts(normalize=True) * 100).round(2))

Churn target distribution (exited):
exited
0    7963
1    2037
Name: count, dtype: int64

Churn target distribution (percentage):
exited
0    79.63
1    20.37
Name: proportion, dtype: float64

A/B conversion distribution (converted):
converted
False    573258
True      14843
Name: count, dtype: int64

A/B conversion distribution (percentage):
converted
False    97.48
True      2.52
Name: proportion, dtype: float64

A/B treatment group distribution (test_group):
test_group
ad     564577
psa     23524
Name: count, dtype: int64

A/B treatment group distribution (percentage):
test_group
ad     96.0
psa     4.0
Name: proportion, dtype: float64


In [11]:
"""
Inspect important categorical variables to understand the composition
of each dataset.
"""

categorical_value_summary(
    churn_df,
    columns=["geography", "gender", "hascrcard", "isactivemember", "exited"]
)

print("\n" + "=" * 80)

categorical_value_summary(
    ab_df,
    columns=["test_group", "converted", "most_ads_day"]
)


Value counts for 'geography':
geography
France     5014
Germany    2509
Spain      2477
Name: count, dtype: int64

Value counts for 'gender':
gender
Male      5457
Female    4543
Name: count, dtype: int64

Value counts for 'hascrcard':
hascrcard
1    7055
0    2945
Name: count, dtype: int64

Value counts for 'isactivemember':
isactivemember
1    5151
0    4849
Name: count, dtype: int64

Value counts for 'exited':
exited
0    7963
1    2037
Name: count, dtype: int64


Value counts for 'test_group':
test_group
ad     564577
psa     23524
Name: count, dtype: int64

Value counts for 'converted':
converted
False    573258
True      14843
Name: count, dtype: int64

Value counts for 'most_ads_day':
most_ads_day
Friday       92608
Monday       87073
Sunday       85391
Thursday     82982
Saturday     81660
Wednesday    80908
Tuesday      77479
Name: count, dtype: int64


In [12]:
"""
Review summary statistics for numeric columns.

This helps identify possible outliers, unrealistic values,
and scale differences across variables.
"""

print("Churn dataset numeric summary:")
display(churn_df.describe())

print("\nA/B testing dataset numeric summary:")
display(ab_df.describe())

Churn dataset numeric summary:


,rownumber,customerid,creditscore,age,tenure,balance,numofproducts,hascrcard,isactivemember,estimatedsalary,exited
count,10000.00000,1.000000e+04,10000.000000,10000.000000,10000.000000,10000.000000,10000.000000,10000.00000,10000.000000,10000.000000,10000.000000
mean,5000.50000,1.569094e+07,650.528800,38.921800,5.012800,76485.889288,1.530200,0.70550,0.515100,100090.239881,0.203700
std,2886.89568,7.193619e+04,96.653299,10.487806,2.892174,62397.405202,0.581654,0.45584,0.499797,57510.492818,0.402769
min,1.00000,1.556570e+07,350.000000,18.000000,0.000000,0.000000,1.000000,0.00000,0.000000,11.580000,0.000000
25%,2500.75000,1.562853e+07,584.000000,32.000000,3.000000,0.000000,1.000000,0.00000,0.000000,51002.110000,0.000000
50%,5000.50000,1.569074e+07,652.000000,37.000000,5.000000,97198.540000,1.000000,1.00000,1.000000,100193.915000,0.000000
75%,7500.25000,1.575323e+07,718.000000,44.000000,7.000000,127644.240000,2.000000,1.00000,1.000000,149388.247500,0.000000
max,10000.00000,1.581569e+07,850.000000,92.000000,10.000000,250898.090000,4.000000,1.00000,1.000000,199992.480000,1.000000



A/B testing dataset numeric summary:


,unnamed:_0,user_id,total_ads,most_ads_hour
count,588101.000000,5.881010e+05,588101.000000,588101.000000
mean,294050.000000,1.310692e+06,24.820876,14.469061
std,169770.279668,2.022260e+05,43.715181,4.834634
min,0.000000,9.000000e+05,1.000000,0.000000
25%,147025.000000,1.143190e+06,4.000000,11.000000
50%,294050.000000,1.313725e+06,13.000000,14.000000
75%,441075.000000,1.484088e+06,27.000000,18.000000
max,588100.000000,1.654483e+06,2065.000000,23.000000


In [13]:
"""
Document initial observations from the ingestion and quality checks.

This is useful for keeping a clear record of early findings
before feature engineering and modeling begin.
"""

quality_observations = [
    "The bank churn dataset contains identifier columns such as rownumber, customerid, and surname that should not be used for predictive modeling.",
    "The churn target variable is 'exited', which is suitable for binary classification.",
    "The A/B testing dataset contains a treatment assignment column ('test_group') and an experimental outcome column ('converted'), making it suitable for A/B testing analysis.",
    "The two datasets are not row-level linked, which is expected. They will be used conceptually together as part of a retention decision framework.",
    "Missing values and duplicates should be reviewed carefully before downstream analysis."
]

for i, observation in enumerate(quality_observations, start=1):
    print(f"{i}. {observation}")

1. The bank churn dataset contains identifier columns such as rownumber, customerid, and surname that should not be used for predictive modeling.
2. The churn target variable is 'exited', which is suitable for binary classification.
3. The A/B testing dataset contains a treatment assignment column ('test_group') and an experimental outcome column ('converted'), making it suitable for A/B testing analysis.
4. The two datasets are not row-level linked, which is expected. They will be used conceptually together as part of a retention decision framework.
5. Missing values and duplicates should be reviewed carefully before downstream analysis.


In [14]:
"""
Save standardized versions of the datasets to the interim folder.

These files preserve the raw data while giving us consistent columns
for future notebooks and scripts.
"""

DATA_INTERIM_DIR = PROJECT_ROOT / "data" / "interim"
DATA_INTERIM_DIR.mkdir(parents=True, exist_ok=True)

CHURN_INTERIM_PATH = DATA_INTERIM_DIR / "bank_churn_standardized.csv"
AB_INTERIM_PATH = DATA_INTERIM_DIR / "marketing_ab_test_standardized.csv"

churn_df.to_csv(CHURN_INTERIM_PATH, index=False)
ab_df.to_csv(AB_INTERIM_PATH, index=False)

print("Saved churn interim file to:", CHURN_INTERIM_PATH)
print("Saved A/B interim file to:", AB_INTERIM_PATH)

Saved churn interim file to: C:\retention_ds\retention_ds\data\interim\bank_churn_standardized.csv
Saved A/B interim file to: C:\retention_ds\retention_ds\data\interim\marketing_ab_test_standardized.csv
